# Testing and Comparing SupHopf Model Connectivity Implementations in TVB and Neurolib

Because connectivity has no fixed standards for how it should be defined, the main painpoint in using new model implementations is the guessing game on how the rows and columns are arranged. To ensure the model is used correctly, a simple way to compare this is necessary. As connectivity includes the tract lengths, which influence the system behavior with a delay, a way to test this part of is also included in this notebook.

This notebook focuses on testing the handling of connectivity and delays in the normal form of a supercritical Hopf bifurcation in Cartesian coordinates<sup>[1]</sup>.

The requirements are as in all other notebooks. Choose a ground truth model to compare against, ensure both models can run one step of Euler integration of size Dt, and that both models can be initialized in the same manner.

The first step is identifying the relevant `dfun` parameters, and setting them to the same values. For TVB and Neurolib, there are only 2 parameters for both models, `a` which appears in both models, and `omega` in TVB which maps to `w` in neurolib.

There are other model parameters relevant for simulation equivalence testing, such as Dt, conduction speed, initial values etc. For the reason of reproducibility we recommend writing a Config class, and a wrapper for each model, so that inputs and outputs match for each model. These can be viewed in files `config.py`, `tvb_model.py` and `neurolib_model.py`.

For parameters relevant to the computation that only appear in one model but not the other, choose values that allow consistent mapping between the models. (Good candidates include powers of ten for scaling, zero for additive terms, or identical "magic constants".)

-------------------

[1] Paraphrased from TVB documentation of SupHopf model.

## Testing of connectivity

The idea behind testing connectivity handling of a simulator is relatively straightforward. Suppose that there are two equivalent models written in C, except one of them uses 32bit floats to represent their floating point numbers, and the other uses 64bit doubles. Given enough steps, these models, although equivalent otherwise, will diverge because of rounding errors, giving us a false negative on the models being equivalent.

To avoid this issue, we've come up with the solution of running only one Euler step at a time, and comparing with rounding errors in mind. If those results are close enough, we know the predicive power of the models is equivalent. We can make this assumption because the Euler step is the simplest possible integration scheme, so if the models diverge on that, they will diverge using any more complex integration.

Keep in mind that while the language of this tutorial talks in absolutes, such as "equivalent", or "same results", we always mean them in the relative sense, in order to avoid awkward phrasing such as "results are within error margins".

## Defining Test Cases

This notebooks assumes that dfun is either correctly implemented or you have already tested it sufficiently. Because of this, the notebook will focus on parameters that relate to connectivity and delays only. These are coupling strength, tract length maximum, conduction speed, and Dt.

Connectivity is tested by setting the coupling strength to a non-zero value, and then comparing the results of the two models. The coupling strength is the only parameter that affects the connectivity, so if the models are equivalent, they should produce the same results when the coupling strength is set to the same value.

Delays are affected by several parameters, including conduction speed, maximum tract length, and the integration time step Dt. For this reason, we opted to do a parameter sweep across common values of Dt and conduction speed. For tract lengths, we opted to normalize them to a known value during config initialization, so that we would always include all delay tracts, and not avoid them using short initial conditions.

## Compatibility Layer

As with other tutorials in this suite, for unification and reproducibility we reccomend building a Config class, as seen below in a simplified version. The relevant parameters for connectivity/delay testing have already been mentioned above, so we point your attention towards the two functions that prepare connectivity (weight and delay matrices most importantly) for testing.

In [1]:
from config import Config
c = Config(initial_conditions_seed=42)
print(f"dt = {c.dt}")
print(f"speed = {c.speed}")
print(f"history_length = {c.history_length}")
print(f"coupling_strength = {c.coupling_strength}")

ModuleNotFoundError: No module named 'config'

In [45]:
import numpy as np
import tvb.simulator.lab as tvbl
from tvb.simulator.models.oscillator import SupHopf
from conn_no_warning import ConnNoWarnings


class ConfigSimplified:
    def __init__(self, initial_conditions_seed, noise_seed=42):
        # other parameters...
        self.dt = 0.001
        self.speed = 6.9
        self.init_cond_rng = np.random.default_rng(seed=initial_conditions_seed)
        self.history_length = 10
        self.coupling_strength = 0.01
        self.conn = None

    def __config_connectivity(self):
        pass

    def init_config_for_connectivity(self):
        self.__config_connectivity()
        self.conn.tract_lengths = np.zeros_like(self.conn.weights)  # because of neurolib
        self.init_cond = np.r_[[[self.init_cond_rng.random((self.size, 1)), self.init_cond_rng.random((self.size, 1))]]]

    def init_config_for_delays(self):
        self.__config_connectivity()
        max_len = np.max(self.conn.tract_lengths)
        self.conn.tract_lengths /= max_len
        self.conn.tract_lengths *= self.history_length - 1
        self.conn.speed = np.r_[self.speed]
        self.init_cond = self.init_cond_rng.random(self.get_good_history_shape())
        self.conn.configure()

    def get_good_history_shape(self):
        self.__config_connectivity()
        delays = self.conn.tract_lengths / self.speed
        idelays = np.rint(delays / self.dt).astype(np.int32)
        init_hist_shape = np.max(idelays) + 1
        init_cond_shape = (init_hist_shape, 2, self.size, 1)
        return init_cond_shape


## Wrappers

As with the previous tutorials, we also need wrappers that have `Config` as input and produce an output relevant to comparison of models.

In [46]:
import numpy as np
from tvb.simulator import simulator, coupling
from tvb.simulator.integrators import EulerDeterministic, EulerStochastic
from tvb.simulator.monitors import Raw
from tvb.simulator.models.oscillator import SupHopf
from config import Config
import tvb.simulator.lab as tvbl

class TvbModel:
    def __init__(self, config: Config):
        self.config = config
        self._configure_sim()

    def _configure_sim(self):
        self.sim = simulator.Simulator(
            connectivity=self.config.conn,
            model=SupHopf(a=np.r_[self.config.a], omega=np.r_[self.config.w]),
            integrator=EulerStochastic(
                dt=self.config.dt,
                noise=tvbl.noise.Additive(
                    nsig=np.r_[self.config.noise],
                    noise_seed=self.config.noise_seed,
                ),
            ),
            initial_conditions=self.config.init_cond,
            conduction_speed=self.config.speed,
            monitors=[Raw()],
            simulation_length=self.config.dt,
            coupling=coupling.Scaling(a=np.r_[self.config.coupling_strength]),
        )
        self.sim.configure()

    def run(self):
        return self.sim.run()[0][1].squeeze()

In [49]:
import numpy as np
from config import Config
from neurolib.models.hopf import HopfModel


class NeurolibModel:
    def __init__(self, config: Config):
        self.config = config
        self._configure_sim()

    def _configure_sim(self):
        self.model = HopfModel(
            Cmat=self.config.conn.weights,
            Dmat=self.config.conn.tract_lengths,
        )
        self.model.params["dt"] = self.config.dt
        self.model.params["xs_init"] = self.config.init_cond[:, 0, :, 0].T
        self.model.params["ys_init"] = self.config.init_cond[:, 1, :, 0].T
        self.model.params["duration"] = self.config.dt
        self.model.params["a"] = self.config.a
        self.model.params["w"] = self.config.w
        self.model.params["coupling"] = "additive"
        self.model.params["K_gl"] = self.config.coupling_strength
        self.model.params["signalV"] = self.config.speed
        if self.config.noise != 0:
            self.model.params["sigma_ou"] = self.config.noise
            self.model.params["x_ou_mean"] = self.config.noise
            self.model.params["y_ou_mean"] = self.config.noise
            self.model.params["x_ou"] = np.random.uniform(
                -self.config.noise,
                self.config.noise,
                (self.config.size,),
            )
            self.model.params["y_ou"] = np.random.uniform(
                -self.config.noise,
                self.config.noise,
                (self.config.size,),
            )

    def run(self):
        self.model.run()
        return self.model.x.squeeze()


## The test itself

As connectivity is affected only by the weights matrix and coupling strength, we do a parameter sweep across several coupling strengths. You are encouraged to replace connectivity matrix with something measured in the real world, as that will better represent the actual use case. For the purpose of these tests we are using the default tutorial connectivity which has been altered to remove symmetry from it.

In [55]:
def run_test(config):
    neurolib_result = NeurolibModel(config).run()
    tvb_result = TvbModel(config).run()
    np.testing.assert_allclose(neurolib_result, tvb_result, atol=1e-6)


def connectivity_test(number_of_tests):
    for i in range(number_of_tests):
        for coupling_strength in range(1, 4):
            config = Config(initial_conditions_seed=i)
            config.coupling_strength = coupling_strength
            print(f"Test {i+1:04d}, coupling strength = {coupling_strength}", end="\r")
            config.init_config_for_connectivity()
        run_test(config)

connectivity_test(100)

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=1e-06

Mismatched elements: 1 / 76 (1.32%)
Max absolute difference: 2.9155012e-06
Max relative difference: 2.54614555e-07
 x: array([ 3.952064,  5.263684,  1.025865,  8.455477,  6.481788,  3.018382,
        7.625266,  7.809876,  4.185405,  0.845566,  9.64096 ,  7.736014,
        5.315004,  6.504622,  7.427357,  6.556178,  8.901128,  9.62232 ,...
 y: array([ 3.952063,  5.263684,  1.025865,  8.455477,  6.481789,  3.018382,
        7.625266,  7.809875,  4.185405,  0.845566,  9.640961,  7.736015,
        5.315004,  6.504621,  7.427357,  6.556179,  8.901129,  9.62232 ,...

## Delay testing

For delay testing the only notable difference from the other tests is that the initial conditions must include a **history** of past states. Because signals travel along tracts, the system’s current state depends on values from several time steps ago. Therefore we have to initialise not just the present state but also the required number of preceding states before we can compute the next integration step.

For the test, we fix the maximum tract length and do a parameter sweep across relevant parameters of the simulation, in this case coupling strength, conduction speed and Dt. For each combination, we test ten different randomized initial conditions as a sanity check.

In [57]:
from itertools import product

dt_options = [0.001, 0.01, 0.1]
c_s_options = [0.01, 0.5, 1.0]
speed_options = list(range(1, 25, 3))

def delay_test():
    for dt, c_s, speed in product(dt_options, c_s_options, speed_options):
        for i in range(10):
            config_delay = Config(initial_conditions_seed=i)
            config_delay.dt = dt
            config_delay.speed = speed
            config_delay.coupling_strength = c_s
            print(f"Test {i+1:02d} with dt={config_delay.dt:.1e}, speed={config_delay.speed:02d}", end="\r")
            config_delay.init_config_for_delays()
            run_test(config_delay)

delay_test()

## On the Pitfalls of Delay Testing

While with connectivity itself we don't have to worry about much, except for if our rows aren't columns on accident, with delays it is more involved. Sometimes the arrays containing history are just initial conditions extended by one direction, like in TVB. But it is feasible to imagine a history array per node, of only the maximum delay length to save memory. In any case, you get no guarantees how a particular implementation is going to handle history arrays larger than the exact required size. So keep that in mind, and be mindful of your history shape. The `get_good_history_shape` function in `Config` class should be easily rewritable if you're not using TVB.